In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))   # notebooks/ -> src/
from data_loader import INPUT_DIR, WORKSPACE, load_long, load_variables

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from models import get_models
from evaluation import BORDERLINE, evaluate, evaluate_lstm_ae, results_table

In [ ]:
# DATASET SWITCH: change this one line, then Run All
SWITCH = "A"  # "A" or "B"

FEATURES = load_variables(SWITCH)                              # sensor columns for this process
BORDERLINE_P = [b for b in BORDERLINE if b.startswith(SWITCH)]   # only process A has borderline batches
print(f"process {SWITCH}: {len(FEATURES)} features, borderline = {BORDERLINE_P}")

In [ ]:
# reading the data: parquet export in team-workspace (~0.5 s; the raw CSV takes ~60 s)
# columns: batch_id, t (0-based step within each batch), then the features as float32
df = load_long(SWITCH, "calibration")
df.head()

In [ ]:
# checking for missing values with count
df.describe().T

In [ ]:
# reading the labels 
df_labels = pd.read_csv(INPUT_DIR / f"process {SWITCH}/calibration data/calibration_labels.csv")

# Brief: A_101-A_104 are REJECTED but borderline. The labels file disagrees (A_101 and A_103 are
# "accepted"), so we override them to "rejected" here. They also get group = "borderline" further down.
# A_102 is only in the test data, so it doesn't appear here.
fix = df_labels["batch_id"].isin(BORDERLINE_P)
print("relabelled as rejected (borderline):",
      df_labels.loc[fix & (df_labels["label"] != "rejected"), "batch_id"].tolist())
df_labels.loc[fix, "label"] = "rejected"
df_labels.head()

In [ ]:
# checking for missing values in the labels
df_labels.describe().T

In [ ]:
# seeing whether the data is time series 
import matplotlib.pyplot as plt
import seaborn as sns

df["batch_id"].value_counts().plot.bar(); plt.show()
df.groupby("batch_id").size().plot.hist(bins=30); plt.title("rows per batch"); plt.show()

In [ ]:
# merging the data and the labels
df = df.merge(df_labels, on="batch_id", how="left")
df["label"].isna().sum()   # rows that didn't get a label: should be 0, or explainable

In [ ]:
# inspecting the new merged data
df.head()

In [ ]:
# "t" (0, 1, 2, ... restarting for each batch) already comes with load_long
df[["batch_id", "t"]].drop_duplicates("batch_id").head()

In [ ]:
df.head()

In [ ]:
# counting the time series length of each batch
lengths = df.groupby("batch_id").size()
lengths.describe()
lengths.value_counts().head()

In [ ]:
# creating a dataframe with one row per batch, making features, creating "borderline" batches
import numpy as np

features = FEATURES

batch_feats = df.groupby("batch_id")[features].agg(["mean", "std", "min", "max"])
batch_feats.columns = ["_".join(c) for c in batch_feats.columns]

t_c = df["t"] - df.groupby("batch_id")["t"].transform("mean")
x_c = df[features] - df.groupby("batch_id")[features].transform("mean")
slopes = x_c.mul(t_c, axis=0).groupby(df["batch_id"]).sum().div(
    (t_c ** 2).groupby(df["batch_id"]).sum(), axis=0)
slopes.columns = [f"{f}_slope" for f in features]

batch_feats = batch_feats.join(slopes).join(df_labels.set_index("batch_id"))

# mark the borderline batches from the brief (label stays "rejected", group = "borderline")
batch_feats["group"] = batch_feats["label"]
batch_feats.loc[batch_feats.index.isin(BORDERLINE_P), "group"] = "borderline"

print(batch_feats.shape)
batch_feats["group"].value_counts()

In [ ]:
# inspecting the time series, plotting. good batches are in blue, bad are in red
import matplotlib.pyplot as plt

ncols = 4
nrows = -(-len(features) // ncols)   # enough rows for every feature (11 for A, 20 for B)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3 * nrows))
for ax, f in zip(axes.flat, features):
    for b, g in df.groupby("batch_id"):
        rej = g["label"].iloc[0] == "rejected"
        ax.plot(g["t"][::10], g[f][::10], color="red" if rej else "blue",
                alpha=0.5 if rej else 0.15, lw=0.7)
    ax.set_title(f)
for ax in axes.flat[len(features):]:
    ax.set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# creating a PCA map of all batches 
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import seaborn as sns

feat_cols = [c for c in batch_feats.columns if c not in ["label", "group"]]
X = batch_feats[feat_cols]
acc = batch_feats["label"] == "accepted"

scaler = StandardScaler().fit(X[acc])      # fit on accepted only
pca = PCA(n_components=2).fit(scaler.transform(X[acc]))
Z = pca.transform(scaler.transform(X))

sns.scatterplot(x=Z[:, 0], y=Z[:, 1], hue=batch_feats["group"], s=50)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%})")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%})")
plt.title("Each dot = one batch"); plt.show()

In [ ]:
# leave-one-batch-out anomaly detection (see evaluation.evaluate)
# The model only ever learns from accepted batches.
# Each accepted batch gets scored by a model that never saw it.
# Rejected batches are scored by every fold's model and averaged.
from sklearn.metrics import roc_auc_score

acc_scores, rej_scores = evaluate(get_models()["Isolation Forest"], X, acc)

scores = pd.concat([acc_scores, rej_scores])       # higher = more anomalous
y = (batch_feats.loc[scores.index, "label"] == "rejected").astype(int)
print("ROC AUC:", round(roc_auc_score(y, scores), 3))

In [ ]:
# picking a threshold and counting the mistakes
threshold = acc_scores.quantile(0.70)   # allows ~30% false alarms on good batches (0.95 -> ~5%)

pred = (scores > threshold).astype(int)
tp = ((pred == 1) & (y == 1)).sum(); fn = ((pred == 0) & (y == 1)).sum()
fp = ((pred == 1) & (y == 0)).sum(); tn = ((pred == 0) & (y == 0)).sum()
print(f"Caught {tp} of {tp+fn} rejected batches, missed {fn}")
print(f"False alarms: {fp} of {fp+tn} accepted batches")

plt.hist(acc_scores, bins=30, alpha=0.6, label="accepted")
plt.hist(rej_scores, bins=30, alpha=0.6, label="rejected", color="red")
plt.axvline(threshold, color="k", ls="--", label="threshold")
plt.xlabel("anomaly score"); plt.legend(); plt.show()

print(scores[scores.index.isin(BORDERLINE_P)])  # where do the borderline ones land?

In [ ]:
# compare all tabular detectors (models.get_models) with the same leave-one-batch-out scheme
import warnings; warnings.filterwarnings("ignore")

all_scores = {}
for name, model in get_models().items():
    print("running", name, "...")
    all_scores[name] = evaluate(model, X, acc)

results_df = results_table(all_scores)
results_df.style.format(precision=3)

In [ ]:
from sklearn.metrics import roc_curve

for name, (acc_s, rej_s) in all_scores.items():
    scores = pd.concat([acc_s, rej_s])
    y = scores.index.isin(rej_s.index).astype(int)
    fpr, tpr, _ = roc_curve(y, scores)
    plt.plot(fpr, tpr, label=f"{name} ({results_df.loc[name, 'ROC AUC']:.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=0.8)
plt.xlabel("false alarm rate"); plt.ylabel("detection rate"); plt.legend(fontsize=8)
plt.title(f"Leave-one-batch-out ROC, process {SWITCH}"); plt.show()

In [ ]:
# LSTM autoencoder on the raw time series (resampled to 100 steps), 5-fold over accepted batches
all_scores["LSTM autoencoder"] = evaluate_lstm_ae(df, batch_feats["label"], features)

results_df = results_table(all_scores)
results_df.style.format(precision=3)

In [ ]:
def leaderboard_f1(acc_s, rej_s, thr, positive="accepted", w_rej=1.0):
    """F1 as the leaderboard computes it. score > thr → predicted rejected."""
    acc_flag, acc_ok = (acc_s > thr).sum(), (acc_s <= thr).sum()
    rej_flag, rej_ok = w_rej * (rej_s > thr).sum(), w_rej * (rej_s <= thr).sum()
    if positive == "accepted":
        tp, fp, fn = acc_ok, rej_ok, acc_flag
    else:
        tp, fp, fn = rej_flag, acc_flag, rej_ok
    return 2 * tp / (2 * tp + fp + fn)

def best_threshold(acc_s, rej_s, positive, test_rej_share):
    n_acc, n_rej = len(acc_s), len(rej_s)
    w = (test_rej_share / (1 - test_rej_share)) * (n_acc / n_rej)   # match test class balance
    cands = np.unique(pd.concat([acc_s, rej_s]))
    f1s = [leaderboard_f1(acc_s, rej_s, t, positive, w) for t in cands]
    i = int(np.argmax(f1s))
    return cands[i], f1s[i], (acc_s <= cands[i]).mean()

for name, (a, r) in all_scores.items():
    for positive, share in [("accepted", 0.26), ("rejected", 0.74)]:
        t, f1, q = best_threshold(a, r, positive, share)
        print(f"{name:<22} positive={positive:<9} best F1 {f1:.3f}  (accepted-quantile {q:.2f})")

In [ ]:
def f1_curve(a, r, positive, share):
    w = (share / (1 - share)) * (len(a) / len(r))
    cands = np.unique(pd.concat([a, r]))
    return cands, np.array([leaderboard_f1(a, r, t, positive, w) for t in cands])

name = "One-class SVM (SGD)"          # swap in "PCA T²" once it's added
a, r = all_scores[name]
c, f_acc = f1_curve(a, r, "accepted", 0.26)
_, f_rej = f1_curve(a, r, "rejected", 0.74)
q = np.array([(a <= t).mean() for t in c])

safe = int(np.argmax(np.minimum(f_acc, f_rej)))
print(f"safe threshold {c[safe]:.4f} (quantile {q[safe]:.2f}): "
      f"F1 {f_acc[safe]:.3f} if accepted is positive, {f_rej[safe]:.3f} if rejected is positive")

plt.plot(q, f_acc, label="positive = accepted")
plt.plot(q, f_rej, label="positive = rejected")
plt.axhline(0.819, ls=":", c="grey", label="current test F1")
plt.axvline(q[safe], ls="--", c="k", lw=0.8, label="safe threshold")
plt.xlabel("threshold (quantile of accepted scores)"); plt.ylabel("F1, process A")
plt.legend(); plt.show()

In [ ]:
def extra_features(df, features):
    g = df.groupby("batch_id")

    # phase means: what happens at the start / middle / end of the batch
    t_norm = df["t"] / g["t"].transform("max")
    phase = pd.cut(t_norm, [-0.01, 0.2, 0.8, 1.0], labels=["start", "mid", "end"])
    ph = df.groupby(["batch_id", phase], observed=True)[features].mean().unstack()
    ph.columns = [f"{f}_{p}" for f, p in ph.columns]

    # robust range: less sensitive to single glitches than min/max
    q = g[features].quantile([0.05, 0.95]).unstack()
    q.columns = [f"{f}_q{int(p * 100)}" for f, p in q.columns]

    # roughness: how jumpy the signal is step to step (your missed temperature_1 batches!)
    same = df["batch_id"].eq(df["batch_id"].shift())
    rough = df[features].diff().where(same).abs().groupby(df["batch_id"]).mean()
    rough.columns = [f"{f}_rough" for f in features]

    # worst local instability: max of a 50-step rolling std
    vol = g[features].rolling(50).std().groupby(level=0).max()
    vol.columns = [f"{f}_maxvol" for f in features]

    # cross-sensor relationships: faults often break correlations, not ranges
    c = g[features].corr().unstack()
    pairs = [(a, b) for i, a in enumerate(features) for b in features[i + 1:]]
    corr = c[pairs].fillna(0)
    corr.columns = [f"corr_{a}__{b}" for a, b in pairs]

    return ph.join(q).join(rough).join(vol).join(corr)

X_extra = extra_features(df, features)
X2 = X.join(X_extra)
print(X.shape, "→", X2.shape)

In [ ]:
L = 100
groups = dict(tuple(df.groupby("batch_id")))
ids = X.index
S = np.stack([np.stack([blk.mean(axis=0) for blk in np.array_split(groups[b][features].to_numpy(), L)])
              for b in ids])                                   # (batches, 100, sensors)
X_traj = pd.DataFrame(S.reshape(len(ids), -1), index=ids)      # one column per (time step, sensor)

In [ ]:
from sklearn.base import BaseEstimator
from sklearn.decomposition import PCA
import numpy as np

class PCAMonitor(BaseEstimator):
    """Multivariate statistical process control: Hotelling T² + Q (SPE) residual."""
    def __init__(self, n_components=10, stat="combined"):
        self.n_components = n_components
        self.stat = stat
    def _raw(self, X):
        X = np.asarray(X); T = self.pca_.transform(X)
        t2 = (T ** 2 / self.pca_.explained_variance_).sum(axis=1)
        q = ((X - self.pca_.inverse_transform(T)) ** 2).sum(axis=1)
        return t2, q
    def fit(self, X, y=None):
        self.pca_ = PCA(self.n_components).fit(np.asarray(X))
        t2, q = self._raw(X)
        self.t2_med_, self.q_med_ = np.median(t2), np.median(q)
        return self
    def score_samples(self, X):
        t2, q = self._raw(X)
        t2, q = t2 / self.t2_med_, q / self.q_med_
        return -{"T2": t2, "Q": q, "combined": t2 + q}[self.stat]

class TopKZ(BaseEstimator):
    """Mean of the k largest robust z-scores (median/MAD, from good batches)."""
    def __init__(self, k=3):
        self.k = k
    def fit(self, X, y=None):
        X = np.asarray(X)
        self.med_ = np.median(X, axis=0)
        self.mad_ = 1.4826 * np.median(np.abs(X - self.med_), axis=0) + 1e-9
        return self
    def score_samples(self, X):
        z = np.abs((np.asarray(X) - self.med_) / self.mad_)
        return -np.sort(z, axis=1)[:, -self.k:].mean(axis=1)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from evaluation import evaluate, results_table

candidates = {
    "T² | base":           (make_pipeline(StandardScaler(), PCAMonitor(10, "T2")), X),
    "T² | +extra":         (make_pipeline(StandardScaler(), PCAMonitor(15, "T2")), X2),
    "T²+Q | +extra":       (make_pipeline(StandardScaler(), PCAMonitor(15, "combined")), X2),
    "OCSVM-SGD | +extra":  (get_models()["One-class SVM (SGD)"], X2),
    "MPCA T² | trajectory":(make_pipeline(StandardScaler(), PCAMonitor(10, "T2")), X_traj),
    "MPCA Q | trajectory": (make_pipeline(StandardScaler(), PCAMonitor(10, "Q")), X_traj),
    "Top-20 z | trajectory": (TopKZ(k=20), X_traj),
}
for name, (model, Xf) in candidates.items():
    print("running", name, "...")
    all_scores[name] = evaluate(model, Xf, acc)

results_table(all_scores).sort_values("PR AUC", ascending=False).style.format(precision=3)

def rank_ensemble(names):
    ranks = pd.concat([pd.concat(all_scores[n]).rank(pct=True) for n in names], axis=1).mean(axis=1)
    a0, r0 = all_scores[names[0]]
    return ranks[a0.index], ranks[r0.index]

all_scores["Ensemble views"] = rank_ensemble(["T² | +extra", "MPCA Q | trajectory", "LSTM autoencoder"])

## hello 

In [ ]:
from sklearn.metrics import average_precision_score

def is_really_better(old, new, n=2000, seed=0):
    a1, r1 = all_scores[old]; a2, r2 = all_scores[new]
    s1 = pd.concat([a1, r1]); s2 = pd.concat([a2, r2]).loc[s1.index]
    y = s1.index.isin(r1.index).astype(int)
    rng = np.random.default_rng(seed); d = []
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    for _ in range(n):
        i = np.r_[rng.choice(pos, len(pos)), rng.choice(neg, len(neg))]
        d.append(average_precision_score(y[i], s2.values[i]) - average_precision_score(y[i], s1.values[i]))
    lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"PR AUC gain {np.mean(d):+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  →",
          "real" if lo > 0 else "could be noise")

is_really_better("One-class SVM (SGD)", "T² | +extra")

In [ ]:
def leaderboard_f1(acc_s, rej_s, thr, positive="accepted", w_rej=1.0):
    """F1 as the leaderboard computes it. score > thr → predicted rejected."""
    acc_flag, acc_ok = (acc_s > thr).sum(), (acc_s <= thr).sum()
    rej_flag, rej_ok = w_rej * (rej_s > thr).sum(), w_rej * (rej_s <= thr).sum()
    if positive == "accepted":
        tp, fp, fn = acc_ok, rej_ok, acc_flag
    else:
        tp, fp, fn = rej_flag, acc_flag, rej_ok
    return 2 * tp / (2 * tp + fp + fn)

def best_threshold(acc_s, rej_s, positive, test_rej_share):
    n_acc, n_rej = len(acc_s), len(rej_s)
    w = (test_rej_share / (1 - test_rej_share)) * (n_acc / n_rej)   # match test class balance
    cands = np.unique(pd.concat([acc_s, rej_s]))
    f1s = [leaderboard_f1(acc_s, rej_s, t, positive, w) for t in cands]
    i = int(np.argmax(f1s))
    return cands[i], f1s[i], (acc_s <= cands[i]).mean()

for name, (a, r) in all_scores.items():
    for positive, share in [("accepted", 0.26), ("rejected", 0.74)]:
        t, f1, q = best_threshold(a, r, positive, share)
        print(f"{name:<22} positive={positive:<9} best F1 {f1:.3f}  (accepted-quantile {q:.2f})")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

POSITIVE = "accepted"     # change to "rejected" if the leaderboard counts rejected as positive

def scores_at_best_threshold(acc_s, rej_s, positive):
    s = pd.concat([acc_s, rej_s])
    is_acc = s.index.isin(acc_s.index)
    y = is_acc.astype(int) if positive == "accepted" else (~is_acc).astype(int)

    best = None
    for t in np.unique(s):
        flagged = (s > t)                                     # above threshold → rejected
        pred = (~flagged).astype(int) if positive == "accepted" else flagged.astype(int)
        f1 = f1_score(y, pred, zero_division=0)
        if best is None or f1 > best[0]:
            best = (f1, pred)
    pred = best[1]
    return {"accuracy": accuracy_score(y, pred),
            "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred, zero_division=0),
            "F1": f1_score(y, pred, zero_division=0)}

table = pd.DataFrame({name: scores_at_best_threshold(a, r, POSITIVE)
                      for name, (a, r) in all_scores.items()}).T
table.sort_values("F1", ascending=False).style.format(precision=3)

In [ ]:
df_test = load_long(SWITCH, "test")
df_test.head()

In [ ]:
def make_features(d):
    bf = d.groupby("batch_id")[features].agg(["mean", "std", "min", "max"])
    bf.columns = ["_".join(c) for c in bf.columns]

    t_c = d["t"] - d.groupby("batch_id")["t"].transform("mean")
    x_c = d[features] - d.groupby("batch_id")[features].transform("mean")
    slopes = x_c.mul(t_c, axis=0).groupby(d["batch_id"]).sum().div(
        (t_c ** 2).groupby(d["batch_id"]).sum(), axis=0)
    slopes.columns = [f"{f}_slope" for f in features]

    return bf.join(slopes).join(extra_features(d, features))

X_test = make_features(df_test)[X2.columns]    # same columns, same order as training
print(X_test.shape, "| missing values:", X_test.isna().sum().sum())

In [ ]:
from sklearn.base import clone

NAME = "T² | +extra"                 # ← the model you picked from your table
model, X_train = candidates[NAME]    # for one of the original models use: get_models()[NAME], X

# 1. final model: trained on ALL accepted calibration batches
final = clone(model).fit(X_train[acc])
test_scores = pd.Series(-final.score_samples(X_test), index=X_test.index)

# 2. threshold: the F1-best value from the leave-one-out scores
a, r = all_scores[NAME]
s = pd.concat([a, r]); is_acc = s.index.isin(a.index)
y = is_acc.astype(int) if POSITIVE == "accepted" else (~is_acc).astype(int)
best_f1, thr = -1, None
for t in np.unique(s):
    pred = (s <= t).astype(int) if POSITIVE == "accepted" else (s > t).astype(int)
    f1 = f1_score(y, pred, zero_division=0)
    if f1 > best_f1:
        best_f1, thr = f1, t
print(f"threshold {thr:.4f} (calibration F1 {best_f1:.3f})")

In [ ]:
submission = pd.DataFrame({
    "batch_id": test_scores.index,
    "label": np.where(test_scores > thr, "rejected", "accepted"),
})
print(submission["label"].value_counts(normalize=True))

# do test scores look like calibration scores? (big shift = warning sign)
plt.hist(a, bins=30, alpha=0.5, density=True, label="calibration accepted")
plt.hist(r, bins=30, alpha=0.5, density=True, label="calibration rejected", color="red")
plt.hist(test_scores, bins=30, alpha=0.5, density=True, label="test", color="green")
plt.axvline(thr, color="k", ls="--", label="threshold"); plt.legend(); plt.show()

submission.to_csv(f"submission_{SWITCH}.csv", index=False)

In [ ]:
bins = np.logspace(np.log10(min(a.min(), r.min(), test_scores.min())),
                   np.log10(max(a.max(), r.max(), test_scores.max())), 40)
plt.hist(a, bins=bins, alpha=0.5, density=True, label="calibration accepted")
plt.hist(r, bins=bins, alpha=0.5, density=True, label="calibration rejected", color="red")
plt.hist(test_scores, bins=bins, alpha=0.5, density=True, label="test", color="green")
plt.axvline(thr, color="k", ls="--", label="threshold")
plt.xscale("log"); plt.xlabel("anomaly score (log scale)"); plt.legend(); plt.show()

print(f"share of calibration accepted above threshold: {(a > thr).mean():.0%}")
print(f"share of test above threshold:                 {(test_scores > thr).mean():.0%}")